# 심층 신경망

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # torch import 전에 설정

from collections import OrderedDict

import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

## seed / determinism

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


## helpers

In [4]:
def load_fashion_mnist():
    tr = datasets.FashionMNIST(root="./data", train=True, download=True)
    te = datasets.FashionMNIST(root="./data", train=False, download=True)
    return (tr.data.numpy(), tr.targets.numpy()), (te.data.numpy(), te.targets.numpy())


def to_loader(x, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g)


def summary(model):
    print(model)
    print("params:", sum(p.numel() for p in model.parameters()))


def fit(model, optimizer, loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
        print(f"Epoch {epoch+1}/{epochs} - loss: {total_loss/n:.4f} - accuracy: {correct/n:.4f}")


def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += criterion(out, yb).item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
    print(f"val loss: {total_loss/n:.4f} - val accuracy: {correct/n:.4f}")
    return total_loss / n, correct / n

In [5]:
criterion = nn.CrossEntropyLoss()  # sparse_categorical_crossentropy + softmax 포함

## 1) 데이터 (flatten 784) + 은닉층 100 sigmoid 모델

In [6]:
(train_input, train_target), (test_input, test_target) = load_fashion_mnist()

train_scaled, test_scaled = train_input / 255.0, test_input / 255.0
train_scaled = train_scaled.reshape(-1, 28*28)

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

print(train_scaled.shape, train_target.shape)
print(val_scaled.shape, val_target.shape)

(48000, 784) (48000,)
(12000, 784) (12000,)


In [7]:
train_loader = to_loader(train_scaled, train_target, shuffle=True)
val_loader = to_loader(val_scaled, val_target, shuffle=False)

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 100), nn.Sigmoid(),
    nn.Linear(100, 10)
).to(device)
summary(model)

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=100, out_features=10, bias=True)
)
params: 79510


이름 지정 버전(name='패션 MNIST 모델'은 PyTorch Sequential에 없으므로 생략)

In [9]:
model = nn.Sequential(OrderedDict([
    ('hidden', nn.Linear(784, 100)),
    ('sigmoid', nn.Sigmoid()),  # ('relu', nn.ReLU()),
    ('output', nn.Linear(100, 10)),
]))
summary(model)

Sequential(
  (hidden): Linear(in_features=784, out_features=100, bias=True)
  (sigmoid): Sigmoid()
  (output): Linear(in_features=100, out_features=10, bias=True)
)
params: 79510


In [10]:
sgd = torch.optim.SGD(params=model.parameters(), lr=0.001, momentum=0.0, nesterov=False)
adagrad = torch.optim.Adagrad(params=model.parameters(), lr=0.01, initial_accumulator_value=0.0, eps=1e-10)
rmsprop = torch.optim.RMSprop(params=model.parameters(), lr=0.01, alpha=0.99, eps=1e-8, momentum=0.0)
adam = torch.optim.Adam(params=model.parameters(), lr=0.001, betas=(0.9, 0.999), eps=1e-8)

In [11]:
optimizer = sgd
fit(model, optimizer, train_loader, epochs=5)

Epoch 1/5 - loss: 2.2630 - accuracy: 0.2960
Epoch 2/5 - loss: 2.1650 - accuracy: 0.5822
Epoch 3/5 - loss: 2.0528 - accuracy: 0.6142
Epoch 4/5 - loss: 1.9208 - accuracy: 0.6301
Epoch 5/5 - loss: 1.7801 - accuracy: 0.6384


In [12]:
optimizer = adagrad
fit(model, optimizer, train_loader, epochs=5)

Epoch 1/5 - loss: 0.5865 - accuracy: 0.8024
Epoch 2/5 - loss: 0.4622 - accuracy: 0.8381
Epoch 3/5 - loss: 0.4324 - accuracy: 0.8478
Epoch 4/5 - loss: 0.4161 - accuracy: 0.8539
Epoch 5/5 - loss: 0.4047 - accuracy: 0.8572


In [13]:
optimizer = rmsprop
fit(model, optimizer, train_loader, epochs=5)

Epoch 1/5 - loss: 0.5599 - accuracy: 0.8013
Epoch 2/5 - loss: 0.4700 - accuracy: 0.8289
Epoch 3/5 - loss: 0.4489 - accuracy: 0.8391
Epoch 4/5 - loss: 0.4444 - accuracy: 0.8425
Epoch 5/5 - loss: 0.4291 - accuracy: 0.8451


In [14]:
optimizer = adam
fit(model, optimizer, train_loader, epochs=5)

Epoch 1/5 - loss: 0.3431 - accuracy: 0.8766
Epoch 2/5 - loss: 0.3174 - accuracy: 0.8843
Epoch 3/5 - loss: 0.3067 - accuracy: 0.8882
Epoch 4/5 - loss: 0.2989 - accuracy: 0.8906
Epoch 5/5 - loss: 0.2909 - accuracy: 0.8926


## 2) Flatten + relu 모델 (입력 28x28)

In [15]:
(train_input, train_target), (test_input, test_target) = load_fashion_mnist()
train_scaled, test_scaled = train_input / 255.0, test_input / 255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

print(train_scaled.shape, train_target.shape)
print(val_scaled.shape, val_target.shape)

(48000, 28, 28) (48000,)
(12000, 28, 28) (12000,)


In [16]:
train_loader = to_loader(train_scaled, train_target, shuffle=True)
val_loader = to_loader(val_scaled, val_target, shuffle=False)

In [17]:
model = nn.Sequential(OrderedDict([
    ('flatten', nn.Flatten()),  # 넣어주는게 무해함
    ('hidden', nn.Linear(784, 100)),
    ('relu', nn.ReLU()),
    ('output', nn.Linear(100, 10)),
]))
summary(model)

Sequential(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=100, out_features=10, bias=True)
)
params: 79510


In [18]:
sgd = torch.optim.SGD(params=model.parameters(), lr=0.001, momentum=0.0, nesterov=False)
adagrad = torch.optim.Adagrad(params=model.parameters(), lr=0.01, initial_accumulator_value=0.0, eps=1e-10)
rmsprop = torch.optim.RMSprop(params=model.parameters(), lr=0.01, alpha=0.99, eps=1e-8, momentum=0.0)
adam = torch.optim.Adam(params=model.parameters(), lr=0.001, betas=(0.9, 0.999), eps=1e-8)

In [19]:
optimizer = sgd
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)

Epoch 1/5 - loss: 1.9850 - accuracy: 0.4715
Epoch 2/5 - loss: 1.3949 - accuracy: 0.6523
Epoch 3/5 - loss: 1.0736 - accuracy: 0.6727
Epoch 4/5 - loss: 0.9200 - accuracy: 0.6956
Epoch 5/5 - loss: 0.8343 - accuracy: 0.7179
val loss: 0.8144 - val accuracy: 0.7222


(0.814418152888616, 0.72225)

In [20]:
optimizer = adagrad
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)

Epoch 1/5 - loss: 0.5118 - accuracy: 0.8237
Epoch 2/5 - loss: 0.4171 - accuracy: 0.8548
Epoch 3/5 - loss: 0.3891 - accuracy: 0.8653
Epoch 4/5 - loss: 0.3722 - accuracy: 0.8705
Epoch 5/5 - loss: 0.3593 - accuracy: 0.8745
val loss: 0.3778 - val accuracy: 0.8659


(0.3778337014516195, 0.8659166666666667)

In [21]:
optimizer = rmsprop
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)

Epoch 1/5 - loss: 0.7519 - accuracy: 0.7938
Epoch 2/5 - loss: 0.4755 - accuracy: 0.8316
Epoch 3/5 - loss: 0.4451 - accuracy: 0.8425
Epoch 4/5 - loss: 0.4311 - accuracy: 0.8474
Epoch 5/5 - loss: 0.4235 - accuracy: 0.8534
val loss: 0.6950 - val accuracy: 0.8041


(0.6949653492371242, 0.8040833333333334)

In [22]:
optimizer = adam
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)

Epoch 1/5 - loss: 0.3299 - accuracy: 0.8797
Epoch 2/5 - loss: 0.3110 - accuracy: 0.8835
Epoch 3/5 - loss: 0.3051 - accuracy: 0.8855
Epoch 4/5 - loss: 0.2996 - accuracy: 0.8882
Epoch 5/5 - loss: 0.2951 - accuracy: 0.8902
val loss: 0.3749 - val accuracy: 0.8718


(0.3749079938530922, 0.8718333333333333)